## 面试问题

循环主控制器怎么写：驱动、注入观察、判断继续/停止？

## 回答主线

控制器是循环的 while 外壳，职责只有编排与裁决，不含业务。本 Notebook 实现一个通用 `loop_controller(init, step_fn, is_done, max_steps)`，用它无改动驱动两个完全不同的任务（退款、猜数），并对比把业务写死在循环里的「胖控制器」——胖控制器换任务即失效。验证瘦控制器可复用、停止信号由控制器统一裁决、终态带停止原因。

## 真实案例

两个任务共用一个控制器：退款任务状态 `paid→verified→refunded`；猜数任务把 `guess` 从 0 递增到 `target`。二者的业务（step 与 done）完全不同，但循环编排一致。数据为教学状态机，不代表真实业务。

In [1]:
def refund_step(state):  # 退款任务的单步业务。
    if state["status"] == "paid":  # 已支付则进入校验。
        return {"status": "verified"}  # 推进到已校验。
    if state["status"] == "verified":  # 已校验则退款。
        return {"status": "refunded"}  # 推进到已退款。
    return state  # 其它状态保持不变。

def refund_done(state):  # 退款完成判定。
    return state["status"] == "refunded"  # 状态为已退款即完成。

def guess_step(state):  # 猜数任务的单步业务。
    return {"guess": state["guess"] + 1, "target": state["target"]}  # 每步把猜测加一。

def guess_done(state):  # 猜数完成判定。
    return state["guess"] >= state["target"]  # 猜到目标即完成。

refund_init = {"status": "paid"}  # 退款任务初始状态。
guess_init = {"guess": 0, "target": 3}  # 猜数任务初始状态。
print("退款初始状态:", refund_init)  # 展示退款任务输入。
print("猜数初始状态:", guess_init)  # 展示猜数任务输入。

退款初始状态: {'status': 'paid'}
猜数初始状态: {'guess': 0, 'target': 3}


## 基线（Baseline）

反面基线：把退款停止条件和状态推进都写死在 while 里的胖控制器。它能跑退款，但停止逻辑与业务耦合，换任务就失效。

In [2]:
def fat_controller_refund(status, max_steps=10):  # 胖控制器：业务与停止判断写死在循环里。
    steps = 0  # 记录步数。
    while status != "refunded":  # 停止条件与退款业务耦合。
        if steps >= max_steps:  # 步数上限兜底。
            break  # 退出循环。
        if status == "paid":  # 业务逻辑混在控制器内。
            status = "verified"  # 推进状态。
        elif status == "verified":  # 业务逻辑混在控制器内。
            status = "refunded"  # 推进状态。
        steps += 1  # 推进步数。
    return status, steps  # 返回状态与步数。

fat_status, fat_steps = fat_controller_refund("paid")  # 运行写死退款的胖控制器。
print("胖控制器退款结果:", fat_status, "| 步数:", fat_steps)  # 展示胖控制器只能服务退款。

胖控制器退款结果: refunded | 步数: 2


## 核心实现：瘦控制器

瘦控制器只做编排：注入 `step_fn` 和 `is_done`，自己持有步数上限，按「完成判定优先、步数上限兜底」裁决停止，返回终态、停止原因、步数和逐步轨迹。

In [3]:
def loop_controller(init_state, step_fn, is_done, max_steps=10):  # 通用循环控制器：只编排不含业务。
    state = init_state  # 从初始状态开始。
    reason = "unknown"  # 记录停止原因。
    steps = 0  # 记录步数。
    trace = []  # 记录每步后的状态快照。
    while True:  # 控制器掌管循环。
        if is_done(state):  # 用注入的完成判定裁决。
            reason = "done"  # 正常完成。
            break  # 退出循环。
        if steps >= max_steps:  # 步数上限兜底。
            reason = "max_steps"  # 超预算停止。
            break  # 退出循环。
        state = step_fn(state)  # 执行注入的单步业务。
        steps += 1  # 推进步数。
        trace.append((steps, dict(state)))  # 追加步号与状态快照。
    return state, reason, steps, trace  # 返回终态、原因、步数、轨迹。

In [4]:
r_state, r_reason, r_steps, r_trace = loop_controller(refund_init, refund_step, refund_done)  # 驱动退款任务并收集轨迹。
g_state, g_reason, g_steps, g_trace = loop_controller(guess_init, guess_step, guess_done)  # 用同一控制器驱动猜数任务。
print("退款任务:", r_state["status"], "| 原因:", r_reason, "| 步数:", r_steps)  # 展示退款任务结果。
for i, snap in r_trace:  # 逐步打印退款轨迹。
    print("  退款 step", i, "->", snap)  # 展示每步状态快照。
print("猜数任务:", g_state["guess"], "| 原因:", g_reason, "| 步数:", g_steps)  # 展示猜数任务结果。
for i, snap in g_trace:  # 逐步打印猜数轨迹。
    print("  猜数 step", i, "->", snap)  # 展示每步状态快照。

退款任务: refunded | 原因: done | 步数: 2
  退款 step 1 -> {'status': 'verified'}
  退款 step 2 -> {'status': 'refunded'}
猜数任务: 3 | 原因: done | 步数: 3
  猜数 step 1 -> {'guess': 1, 'target': 3}
  猜数 step 2 -> {'guess': 2, 'target': 3}
  猜数 step 3 -> {'guess': 3, 'target': 3}


## 结果解读

同一个 `loop_controller` 无改动完成了退款（2 步）和猜数（3 步），停止原因都是 `done`——业务差异全部落在注入的 `step_fn`/`is_done` 里，编排与停止逻辑复用。逐步轨迹显示状态如何单调推进到完成条件。

## 失败案例与修正

把退款胖控制器误用于猜数：它的停止条件 `status != "refunded"` 对猜数状态永不成立，循环只能空转到步数上限。这说明「业务停止条件写死在控制器里」为什么不可复用。修正就是上面的瘦控制器——把 `is_done` 作为参数注入。

In [5]:
def misuse_fat_for_guess(max_steps=10):  # 误把退款胖控制器用于猜数任务。
    status = "guessing"  # 猜数状态不是退款状态。
    steps = 0  # 记录步数。
    while status != "refunded":  # 退款停止条件对猜数永不成立。
        if steps >= max_steps:  # 只能靠步数上限退出。
            break  # 退出循环。
        steps += 1  # 空转推进。
    return status, steps  # 返回状态与步数。

misuse_status, misuse_steps = misuse_fat_for_guess()  # 运行被误用的胖控制器。
print("胖控制器误用于猜数:", misuse_status, "| 空转步数:", misuse_steps)  # 展示停止条件失配导致空转到上限。

胖控制器误用于猜数: guessing | 空转步数: 10


In [6]:
assert r_state["status"] == "refunded"  # 通用控制器应完成退款任务。
assert g_state["guess"] == 3  # 通用控制器应完成猜数任务。
assert r_reason == "done"  # 退款应因完成判定停止。
assert g_reason == "done"  # 猜数应因完成判定停止。
assert misuse_steps == 10  # 误用胖控制器因停止条件失配空转到上限。
assert fat_status == "refunded"  # 胖控制器只在其写死的退款任务上正确。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
